# Cross-Lingual Multimodal Transfer Pipeline

Research question: does adding audio help cross-lingual transfer for laughter-associated humor prediction?

Experiment design in this notebook:

- Train on English text only, evaluate on held-out English plus French / Spanish / Hungarian text.
- Train on English text + audio, evaluate on held-out English plus French / Spanish / Hungarian text + audio.

In [ ]:
%pip install -U torch transformers librosa soundfile scikit-learn pandas numpy tqdm accelerate

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch

from pipeline.config import PipelineConfig
from pipeline.embeddings import load_or_extract_features
from pipeline.experiment import build_crosslingual_dataset
from pipeline.laughter import AnnotationLaughterDetector
from pipeline.train import (
    evaluate_on_indices,
    metric_row,
    train_transfer_model,
    video_train_val_test_indices,
)

## Config

`RMSLaughterDetector` is only the temporary label module.

In [ ]:
cfg = PipelineConfig(
    lang="en",
    # audio_root_name="dataset/audios/en",
    sample_limit=None,  # set to None for the full experiment
    random_seed=42,
    qwen_model_id="Qwen/Qwen3-0.6B",
    whisper_model_id="openai/whisper-large-v3-turbo",
    batch_size_embed_text=4,
    batch_size_train=16,
    epochs=8,
    fusion_hidden_dim=512,
    fusion_num_heads=8,
)

TRAIN_LANG = "en"
TEST_LANGS = ["fr", "es", "hu"]
ALL_LANGS = [TRAIN_LANG] + TEST_LANGS

AUDIO_ROOTS = {
    "en": cfg.project_root / "cleaned-data" / "en",
    "fr": cfg.project_root / "cleaned-data" / "fr",
    "es": cfg.project_root / "cleaned-data" / "es",
    "hu": cfg.project_root / "cleaned-data" / "hu",
}

SAMPLE_LIMITS = {lang: cfg.sample_limit for lang in ALL_LANGS}

MODEL_MODES = {
    "text_only": "text",
    "text_audio_concat": "concat",
    "text_audio_cross_attention": "cross_attention",
}

random.seed(cfg.random_seed)
np.random.seed(cfg.random_seed)
torch.manual_seed(cfg.random_seed)

print("device:", cfg.device)
print("artifacts:", cfg.artifact_dir)
for lang, root in AUDIO_ROOTS.items():
    print(lang, "transcripts=", cfg.project_root / "asr-output" / lang, "audio=", root)

## Build Cross-Lingual Dataset

In [ ]:
laughter_detector = AnnotationLaughterDetector(cfg)
matched_by_lang, segments_df = build_crosslingual_dataset(
    ALL_LANGS,
    AUDIO_ROOTS,
    laughter_detector,
    cfg,
    sample_limits=SAMPLE_LIMITS,
)

segments_csv = cfg.artifact_dir / "crosslingual_segment_dataset.csv"
segments_df.to_csv(segments_csv, index=False)

print("saved:", segments_csv)
print("segments:", segments_df.shape)
print("matched videos by language:")
for lang, matched in matched_by_lang.items():
    print(lang, len(matched))
print("segments by language and label:")
print(pd.crosstab(segments_df["lang"], segments_df["label"]))
segments_df.head()

## English Train/Validation Split

Only English is used for training. English has a held-out in-language test split; French, Spanish, and Hungarian are held out as cross-lingual test languages.

In [ ]:
train_idx, val_idx, en_test_idx = video_train_val_test_indices(
    segments_df,
    TRAIN_LANG,
    cfg.random_seed,
    val_size=0.15,
    test_size=0.15,
)

test_indices = {TRAIN_LANG: en_test_idx}
test_indices.update({
    lang: segments_df.index[segments_df["lang"] == lang].to_numpy()
    for lang in TEST_LANGS
})

print("train segments:", len(train_idx), "val segments:", len(val_idx), "en test segments:", len(en_test_idx))
for lang, idx in test_indices.items():
    print(f"test {lang}:", len(idx))

## Extract Or Load Features

The cache checks row identity, so changing language/sample selection will trigger recomputation automatically.

In [ ]:
text_hidden, audio_hidden, y = load_or_extract_features(segments_df, cfg, force_recompute=False)

print("num segments:", len(y))
print("text dim:", text_hidden[0].shape[-1], "example text shape:", text_hidden[0].shape)
print("audio dim:", audio_hidden[0].shape[-1], "example audio shape:", audio_hidden[0].shape)
print("labels:", y.shape, np.bincount(y))

## Train On English, Test Cross-Lingually

In [ ]:
results = []
histories = {}
trained_models = {}

for model_name, mode in MODEL_MODES.items():
    print("\n===", model_name, "===")
    model, history = train_transfer_model(mode, text_hidden, audio_hidden, y, train_idx, val_idx, cfg)
    trained_models[model_name] = model
    histories[model_name] = history

    for test_lang, indices in test_indices.items():
        metrics = evaluate_on_indices(model, text_hidden, audio_hidden, y, indices, cfg)
        results.append(metric_row(model_name, TRAIN_LANG, test_lang, metrics))

results_df = pd.DataFrame(results).sort_values(["test_lang", "model"])
results_path = cfg.artifact_dir / "crosslingual_transfer_results.csv"
results_df.to_csv(results_path, index=False)
print("saved:", results_path)
results_df


## Reading The Result

The key comparison is within each test language:

```text
text_only vs text_audio_concat / text_audio_cross_attention
```

If text+audio improves F1 on held-out English and/or French, Spanish, or Hungarian relative to text-only, that supports the hypothesis that audio helps cross-lingual transfer.

## Replacing The Label Module

```python
class Standup4AILaughterDetector:
    def detect(self, row):
        return [(start_sec, end_sec), ...]
```

Replace:

```python
laughter_detector = AnnotationLaughterDetector(cfg)
```

